# Load Data

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

# Initial Data Exploration

Perform initial data exploration to understand the dataset structure, data types, missing values, and basic statistics. This step will also identify potential issues for data cleaning.

In [ ]:
print("DataFrame shape:", df.shape)
print("\nDataFrame information:")
df.info()
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nDescriptive statistics for numerical columns:")
print(df.describe())

In [ ]:
# check that there are empty strings in TotalCharges
print("Rows with empty strings in 'TotalCharges':")
print(df[df['TotalCharges'] == ' '])

In [ ]:
# replace values where TotalCharges is an empty string with 0.0 since those are customers with a tenure of 0
df['TotalCharges'] = df['TotalCharges'].replace(' ', 0.0)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
df.info()

In [ ]:
df.to_csv('telecom_churn_cleaned.csv',index=False)

# Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# plot pie chart of churn percent
churn_counts = df['Churn'].value_counts()
print(churn_counts)

plt.figure(figsize=(6, 6))
plt.pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%', startangle=90, colors=['skyblue', 'lightcoral'])
plt.title('Distribution of Churn')
plt.axis('equal')
plt.show()

In [ ]:
#plot histogram of numerical features

numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']

plt.figure(figsize=(15, 5))
for i, feature in enumerate(numerical_features):
    plt.subplot(1, 3, i + 1)
    sns.histplot(df[feature], kde=True)
    plt.title(f'Distribution of {feature}')
    plt.xlabel(feature)
    plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
#bucket monthly charges into bins of 10 and plot churn frequency

bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120]
labels = ['0-10', '11-20', '21-30', '31-40', '41-50', '51-60', '61-70', '71-80', '81-90', '91-100', '101-110', '111-120']
df['monthly_charges_group'] = pd.cut(df['MonthlyCharges'], bins=bins, labels=labels, right=False)

plt.figure(figsize=(10, 6))
sns.countplot(x='monthly_charges_group', hue='Churn', data=df, palette='viridis')
plt.title('Churn by Monthly Charges')
plt.xlabel('Monthly Charges Group ($)')
plt.ylabel('Frequency')
plt.legend(title='Churn', labels=['No', 'Yes'])
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

df = df.drop('monthly_charges_group', axis=1)

In [ ]:
#bucket monthly charges into bins of 10 and plot churn frequency

bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120]
labels = ['0-10', '11-20', '21-30', '31-40', '41-50', '51-60', '61-70', '71-80', '81-90', '91-100', '101-110', '111-120']
df['monthly_charges_group'] = pd.cut(df['MonthlyCharges'], bins=bins, labels=labels, right=False)

plt.figure(figsize=(10, 6))
sns.countplot(x='monthly_charges_group', hue='Churn', data=df, palette='viridis')
plt.title('Higher Monthly Charges Are Associated with Increased Churn')
plt.xlabel('Monthly Charges Group ($)')
plt.ylabel('Customer Count')
plt.legend(title='Churn', labels=['No', 'Yes'])
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

df = df.drop('monthly_charges_group', axis=1)

In [ ]:
# bucket total charges into bins of 100 and plot churn frequency

bins = [0, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000]
labels = ['0-1000', '1001-2000', '2001-3000', '3001-4000', '4001-5000', '5001-6000', '6001-7000', '7001-8000', '8001-9000']
df['total_charges_group'] = pd.cut(df['TotalCharges'], bins=bins, labels=labels, right=False)

plt.figure(figsize=(12, 6))
sns.countplot(x='total_charges_group', hue='Churn', data=df, palette='viridis')
plt.title('Churn by Total Charges')
plt.xlabel('Total Charges Group ($)')
plt.ylabel('Frequency')
plt.legend(title='Churn', labels=['No', 'Yes'])
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

df = df.drop('total_charges_group', axis=1)

In [ ]:
# plot average monthly charges in relation to tenure

avg_monthly_charges_by_tenure = df.groupby('tenure')['MonthlyCharges'].mean().reset_index()

plt.figure(figsize=(10, 6))
sns.lineplot(x='tenure', y='MonthlyCharges', data=avg_monthly_charges_by_tenure)
plt.title('Average Monthly Charges by Tenure')
plt.xlabel('Tenure (Months)')
plt.ylabel('Average Monthly Charges')
plt.grid(True)
plt.show()

In [ ]:
# plot correlation heatmap of numeric features

plt.figure(figsize=(18, 15))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Numeric Features')
plt.show()

In [ ]:
# plot churn of every categorical feature

categorical_churn_features = [
    'gender', 'SeniorCitizen', 'Partner',
    'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV',
    'StreamingMovies', 'Contract', 'PaperlessBilling',
    'PaymentMethod'
]

plt.figure(figsize=(20, 25))
for i, feature in enumerate(categorical_churn_features):
    plt.subplot(6, 3, i + 1)
    sns.countplot(x=feature, hue='Churn', data=df)
    plt.title(f'{feature} vs. Churn')
    plt.xlabel(feature)
    plt.ylabel('Count')
    plt.legend(title='Churn', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()
print("Count plots for selected categorical features against 'Churn' displayed.")

# Machine Learning

## Prepare Data

In [ ]:
# convert Churn to numerical
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# drop customerID
df = df.drop('customerID', axis=1)

In [ ]:
# manually map baseline before one-hot encoding on object columns

baseline_map = {
    "gender": ["Male", "Female"],
    "Partner": ["No", "Yes"],
    "Dependents": ["No", "Yes"],
    "PhoneService": ["No", "Yes"],
    "MultipleLines": ["No", "Yes", "No phone service"],
    "InternetService": ["No", "DSL", "Fiber optic"],
    "OnlineSecurity": ["No", "Yes", "No internet service"],
    "OnlineBackup": ["No", "Yes", "No internet service"],
    "DeviceProtection": ["No", "Yes", "No internet service"],
    "TechSupport": ["No", "Yes", "No internet service"],
    "StreamingTV": ["No", "Yes", "No internet service"],
    "StreamingMovies": ["No", "Yes", "No internet service"],
    "Contract": ["Month-to-month", "One year", "Two year"],
    "PaperlessBilling": ["No", "Yes"],
    "PaymentMethod": [
        "Electronic check",
        "Mailed check",
        "Bank transfer (automatic)",
        "Credit card (automatic)"
    ]
}

for col, categories in baseline_map.items():
    df[col] = pd.Categorical(df[col], categories=categories, ordered=True)

# one-hot encoding
categorical_cols = list(baseline_map.keys())
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print("\nCategorical columns one-hot encoded. New DataFrame shape:", df.shape)
print(df.columns.tolist())

In [ ]:
# plot heatmap of correlation matrix of entire dataframe

plt.figure(figsize=(18, 15))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Features')
plt.show()

In [ ]:
# get model
from sklearn.model_selection import train_test_split

In [ ]:
# separate features (X) and target (y), and further split into training and testing sets with 80/20 ratio for model development

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

## Train Model

In [ ]:
# import
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay


### Random Forest

In [ ]:
# Instantiate the model
rf_model = RandomForestClassifier(random_state=42)

# Train the model
rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully.")

In [ ]:
# Make predictions on the test set
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# Print Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Calculate and print ROC AUC Score
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"\nROC AUC Score: {roc_auc:.4f}")

# Visualize Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
display_cm = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf_model.classes_)
fig, ax = plt.subplots(figsize=(8, 6))
display_cm.plot(ax=ax, cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

# Visualize ROC Curve
fig, ax = plt.subplots(figsize=(8, 6))
roc_display = RocCurveDisplay.from_estimator(rf_model, X_test, y_test, ax=ax, name='Random Forest')
plt.title('ROC Curve')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.legend()
plt.show()


### Linear

In [ ]:
# Instantiate the Linear Regression model
lr_model = LinearRegression()

# Train the model
lr_model.fit(X_train, y_train)

print("Linear Regression model trained successfully.")

In [ ]:
# Make predictions on the test set using the linear model's continuous output
y_pred_lr_continuous = lr_model.predict(X_test)
# Convert continuous predictions to binary using a 0.5 threshold for classification metrics
y_pred_lr_binary = (y_pred_lr_continuous > 0.5).astype(int)

# Print Classification Report
print("Linear Regression Classification Report (0.5 threshold):")
print(classification_report(y_test, y_pred_lr_binary))

# Calculate and print ROC AUC Score using continuous predictions
roc_auc = roc_auc_score(y_test, y_pred_lr_continuous)
print(f"\nLinear Regression ROC AUC Score: {roc_auc:.4f}")

# Visualize Confusion Matrix
cm = confusion_matrix(y_test, y_pred_lr_binary)
display_cm = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
fig, ax = plt.subplots(figsize=(8, 6))
display_cm.plot(ax=ax, cmap='Blues')
plt.title('Linear Regression Confusion Matrix (Threshold = 0.5)')
plt.show()

# Visualize ROC Curve
fig, ax = plt.subplots(figsize=(8, 6))
# Use RocCurveDisplay.from_predictions as LinearRegression doesn't have predict_proba
RocCurveDisplay.from_predictions(y_test, y_pred_lr_continuous, ax=ax, name='Linear Regression')
plt.title('Linear Regression ROC Curve')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.legend()
plt.show()

### Logarithmic

In [ ]:
# Instantiate the Logistic Regression model
log_reg_model = LogisticRegression(random_state=42)

# Train the model
log_reg_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

In [ ]:
# Make predictions on the test set
y_pred_log_reg = log_reg_model.predict(X_test)
y_pred_proba_log_reg = log_reg_model.predict_proba(X_test)[:, 1]

# Print Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred_log_reg))

# Calculate and print ROC AUC Score
roc_auc_log_reg = roc_auc_score(y_test, y_pred_proba_log_reg)
print(f"\nROC AUC Score: {roc_auc_log_reg:.4f}")

# Visualize Confusion Matrix
cm_log_reg = confusion_matrix(y_test, y_pred_log_reg)
display_cm_log_reg = ConfusionMatrixDisplay(confusion_matrix=cm_log_reg, display_labels=log_reg_model.classes_)
fig_cm_log_reg, ax_cm_log_reg = plt.subplots(figsize=(8, 6))
display_cm_log_reg.plot(ax=ax_cm_log_reg, cmap='Blues')
plt.title('Logistic Regression Confusion Matrix')
plt.show()

# Visualize ROC Curve
fig_roc_log_reg, ax_roc_log_reg = plt.subplots(figsize=(8, 6))
roc_display_log_reg = RocCurveDisplay.from_estimator(log_reg_model, X_test, y_test, ax=ax_roc_log_reg, name='Logistic Regression')
plt.title('Logistic Regression ROC Curve')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.legend()
plt.show()


In [ ]:
# change the threshold to 0.4 for the logistic model and try again. aiming to improve recall score

# Make predictions on the test set
y_pred_proba_log_reg_2 = log_reg_model.predict_proba(X_test)[:, 1]
y_pred_log_reg_2 = (y_pred_proba_log_reg_2 > 0.4).astype(int)

# Print Classification Report
print("Classification Report (0.4 threshold):")
print(classification_report(y_test, y_pred_log_reg_2))

# Calculate and print ROC AUC Score
roc_auc_log_reg_2 = roc_auc_score(y_test, y_pred_proba_log_reg_2)
print(f"\nROC AUC Score: {roc_auc_log_reg_2:.4f}")

# Visualize Confusion Matrix
cm_log_reg_2 = confusion_matrix(y_test, y_pred_log_reg_2)
display_cm_log_reg_2 = ConfusionMatrixDisplay(confusion_matrix=cm_log_reg_2, display_labels=log_reg_model.classes_)
fig_cm_log_reg_2, ax_cm_log_reg_2 = plt.subplots(figsize=(8, 6))
display_cm_log_reg_2.plot(ax=ax_cm_log_reg_2, cmap='Blues')
plt.title('Logistic Regression Confusion Matrix (Threshold = 0.4)')
plt.show()

# Visualize ROC Curve
fig_roc_log_reg_2, ax_roc_log_reg_2 = plt.subplots(figsize=(8, 6))
roc_display_log_reg_2 = RocCurveDisplay.from_estimator(log_reg_model, X_test, y_test, ax=ax_roc_log_reg_2, name='Logistic Regression')
plt.title('Logistic Regression ROC Curve')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.legend()
plt.show()


# Predictive Insights Visualiztion

In [ ]:
# get feature importances from the logistic regression model (coefficients)
feature_importance = pd.Series(log_reg_model.coef_[0], index=X_train.columns)
feature_importance = feature_importance.sort_values(key=abs, ascending=False)

print("Top 10 most important features from Logistic Regression (sorted by absolute coefficient value):")
print(feature_importance.head(10))

In [ ]:
# plot top 10 most important features

plt.figure(figsize=(12, 7))

top10 = feature_importance.head(10)
colors = ['red' if c < 0 else 'green' for c in top10.values]

sns.barplot(
    x=top10.index,
    y=top10.values,
    hue=top10.index, 
    palette=colors,
    legend=False   
)

plt.title('Top 10 Most Important Features for Churn Prediction (Logistic Regression Coefficients)')
plt.xlabel('Features')
plt.ylabel('Coefficient Value')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


# Project Summary

This project aims to analyze customer churn using the Telco Customer Churn dataset. There are several stages.

## 1. Data Loading and Initial Exploration
*   The dataset was loaded from a CSV file into a pandas DataFrame.
*   Initial exploration revealed the dataset shape, data types, and confirmed no missing values, except for empty strings in 'TotalCharges', which were converted to 0.0 and the column type was changed to numeric.

## 2. Exploratory Data Analysis (EDA)
*   **Churn Distribution:** The dataset showed an imbalance, with approximately 26.5% of customers churning.
*   **Numerical Features:**
    *   **Tenure:** Customers with shorter tenures (especially 0-10 months) showed a significantly higher churn rate. Churn decreased as tenure increased.
    *   **Monthly Charges:** Customers with higher monthly charges, particularly in the mid-to-high ranges, exhibited a greater propensity to churn.
    *   **Total Charges:** Lower total charges were associated with higher churn, often correlating with shorter tenure.
*   **Categorical Features:** Various categorical features like 'Contract', 'InternetService', 'OnlineSecurity', 'TechSupport', 'PaperlessBilling', and 'PaymentMethod' showed clear relationships with churn.

## 3. Data Preprocessing
*   The 'customerID' column was dropped.
*   The 'Churn' target variable was converted from categorical ('Yes', 'No') to numerical (1, 0).
*   Categorical features were one-hot encoded to convert them into a numerical format suitable for machine learning models.
*   The dataset was split into training (80%) and testing (20%) sets, stratified by the 'Churn' variable.

## 4. Model Training and Evaluation
Three different classification models were trained and evaluated:

### a. Random Forest Classifier
*   **Accuracy:** 0.78
*   **ROC AUC Score:** 0.8219
*   **Recall (Churn Class):** 0.47
    *   Showed good overall performance but struggled with recalling actual churners (false negatives).

### b. Linear Regression (adapted for classification)
*   **Accuracy:** 0.80
*   **ROC AUC Score:** 0.8297
*   **Recall (Churn Class):** 0.52
    *   Achieved competitive results, slightly outperforming Random Forest in terms of ROC AUC and churn recall, despite being a linear model.

### c. Logistic Regression
*   **Accuracy (Default 0.5 threshold):** 0.80
*   **ROC AUC Score:** 0.8415
*   **Recall (Churn Class):** 0.56
    *   This model demonstrated the best overall performance with the highest ROC AUC score and the best recall for the churn class among the three models using the default threshold.
*   **Logistic Regression (0.4 threshold):**
    *   **Accuracy:** 0.78
    *   **Recall (Churn Class):** 0.67
        *   Adjusting the classification threshold to 0.4 significantly improved the recall for the churn class (meaning more actual churners were identified), though this came with a slight decrease in precision (more false positives).

## 5. Predictive Insights
*   **Feature Importance (Logistic Regression):**
    *   Features like **Contract_Two year**, **OnlineSecurity_Yes**, and **TechSupport_Yes** had negative coefficients, indicating they **decrease** the likelihood of churn.
    *   Features like **InternetService_Fiber optic**, **PaperlessBilling_Yes**, and **PaymentMethod_Electronic check** had positive coefficients, indicating they **increase** the likelihood of churn.

## Key Findings & Recommendations
*   **Contract Length & Services:** Customers on longer contracts and those utilizing additional services like online security and technical support are less likely to churn. **Recommendation:** Offer incentives for longer-term contracts and promote value-added services.
*   **Payment & Billing:** Customers using electronic checks and paperless billing showed a higher propensity to churn. **Recommendation:** Investigate the customer experience associated with these methods and potentially offer alternative payment incentives or support.
*   **Early Customer Engagement:** Newer customers (lower tenure) are a high-risk group. **Recommendation:** Implement early engagement programs and monitor satisfaction closely during the initial months.
*   **High Monthly Charges:** Customers with higher monthly charges are more prone to churn. **Recommendation:** Review pricing strategies, offer personalized plans, or provide additional benefits to high-paying customers to justify costs and increase perceived value.
*   **Model Choice:** Logistic Regression with a fine-tuned threshold (e.g., 0.4) emerged as the most effective model for identifying churners, balancing the need for high recall to enable proactive retention efforts.